# Triple-Barrier Labeling — SPY / EUR-USD / Gold

Labels each 5-min bar as `long`, `short`, or `no_trade` based on which of three barriers gets touched first, looking forward from that bar:

- **Upper barrier** = entry + `TP_MULT` x ATR(14) at entry -> touched first = `long`
- **Lower barrier** = entry - `SL_MULT` x ATR(14) at entry -> touched first = `short`
- **Vertical barrier** = `MAX_HOLDING_BARS` bars ahead, or the end of the trading session/before a weekend gap, whichever comes first -> touched first (neither price barrier hit) = `no_trade`

Barriers are ATR-based (not fixed price/pips) so they adapt to each bar's own volatility regime, consistent with how we planned to size SL/TP for the live signal output.

**Design notes:**
- **No overnight/weekend holds**: for SPY, the vertical barrier is capped at session end (no trade label implies holding into the next day). For EUR/USD and Gold, a large time gap between consecutive bars (>4h) is treated as a session break (catches the weekend close) with the same capping logic.
- **Same-bar ambiguity**: if a single forward bar's range touches *both* barriers (a big/gappy bar), we can't know which was hit first from OHLC alone. We approximate using the bar's open: whichever barrier the open is closer to is assumed touched first. This is a real limitation of bar data (vs. tick data) — a known approximation, not a bug.
- **Tail of each dataset is left unlabeled, not force-labeled**: rows near the end of the file don't have enough future bars to know what would have happened. Those get dropped rather than mislabeled as `no_trade`.
- Run this in VSCode on the fixed `features_5min.csv` files — output is `features_5min_labeled.csv` per market, which is what gets uploaded into the three separate Colab modeling notebooks.

## 0. Config

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

MARKETS = {
    "spy":    {"session_based": True},
    "eurusd": {"session_based": False},
    "gold":   {"session_based": False},
}

TP_MULT = 2.5          # take-profit barrier, in multiples of ATR(14) at entry
SL_MULT = 1.5          # stop-loss barrier, in multiples of ATR(14) at entry
MAX_HOLDING_BARS = 24  # vertical barrier: 24 x 5min = 2 hours max hold
SESSION_GAP_HOURS = 4  # gap larger than this (non-session markets) = treat as a session break (weekend)

## 1. Load the fixed feature files

In [ ]:
def load_features(market: str) -> pd.DataFrame:
    path = DATA_DIR / market / "features_5min.csv"
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    df = df.sort_values("datetime").reset_index(drop=True)
    return df

## 2. Triple-barrier labeling function

Pure numpy-array loop for clarity over vectorization tricks. On ~300k rows this may take a minute or two per market — that's expected; it's a one-time offline step, not something that needs to be fast.

In [ ]:
def label_triple_barrier(df: pd.DataFrame, session_based: bool,
                          tp_mult: float, sl_mult: float, max_holding: int) -> pd.DataFrame:
    n = len(df)
    close = df["close"].to_numpy()
    high = df["high"].to_numpy()
    low = df["low"].to_numpy()
    open_ = df["open"].to_numpy()
    atr = df["atr_14"].to_numpy()

    if session_based:
        et_dates = df["datetime"].dt.tz_convert("America/New_York").dt.date.to_numpy()
        session_id = pd.factorize(et_dates)[0]
    else:
        gap_seconds = df["datetime"].diff().dt.total_seconds().fillna(0).to_numpy()
        session_id = np.cumsum(gap_seconds > SESSION_GAP_HOURS * 3600)

    label = np.full(n, np.nan, dtype=object)
    barrier_hit = np.full(n, np.nan, dtype=object)
    exit_idx = np.full(n, -1, dtype=int)
    exit_price = np.full(n, np.nan)
    holding_bars = np.full(n, np.nan)

    for i in range(n):
        a = atr[i]
        if np.isnan(a) or a <= 0:
            continue

        entry = close[i]
        upper = entry + tp_mult * a
        lower = entry - sl_mult * a
        sid = session_id[i]

        # walk forward up to max_holding bars, staying inside the same session
        j = i
        steps = 0
        while steps < max_holding and j + 1 < n and session_id[j + 1] == sid:
            j += 1
            steps += 1
        window_end = j
        hit_full_horizon = steps == max_holding
        ran_out_of_data = (not hit_full_horizon) and (window_end + 1 >= n)

        resolved = False
        for k in range(i + 1, window_end + 1):
            hit_upper = high[k] >= upper
            hit_lower = low[k] <= lower
            if hit_upper and hit_lower:
                # same-bar ambiguity -- approximate using distance from the bar's open
                if (open_[k] - lower) < (upper - open_[k]):
                    label[i], exit_price[i], barrier_hit[i] = "short", lower, "both_same_bar"
                else:
                    label[i], exit_price[i], barrier_hit[i] = "long", upper, "both_same_bar"
                exit_idx[i] = k
                resolved = True
                break
            elif hit_upper:
                label[i], exit_price[i], barrier_hit[i] = "long", upper, "upper"
                exit_idx[i] = k
                resolved = True
                break
            elif hit_lower:
                label[i], exit_price[i], barrier_hit[i] = "short", lower, "lower"
                exit_idx[i] = k
                resolved = True
                break

        if not resolved:
            if ran_out_of_data:
                continue  # true tail of the dataset -- leave unlabeled, dropped later
            label[i] = "no_trade"
            exit_price[i] = close[window_end]
            exit_idx[i] = window_end
            barrier_hit[i] = "timeout" if hit_full_horizon else "session_end"

        if exit_idx[i] != -1:
            holding_bars[i] = exit_idx[i] - i

    out = df.copy()
    out["label"] = label
    out["barrier_hit"] = barrier_hit
    out["exit_idx"] = exit_idx
    out["exit_price"] = exit_price
    out["holding_bars"] = holding_bars
    return out

## 3. Run for all three markets

In [ ]:
labeled = {}

for market, cfg in MARKETS.items():
    print(f"=== {market} ===")
    df = load_features(market)
    out = label_triple_barrier(
        df, session_based=cfg["session_based"],
        tp_mult=TP_MULT, sl_mult=SL_MULT, max_holding=MAX_HOLDING_BARS,
    )

    before = len(out)
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    print(f"  {before:,} -> {len(out):,} rows after dropping unlabeled tail")

    out_path = DATA_DIR / market / "features_5min_labeled.csv"
    out.to_csv(out_path, index=False)
    print(f"  saved -> {out_path}")

    labeled[market] = out

## 4. Check the label distribution and holding stats before moving to Colab

In [ ]:
for market, df in labeled.items():
    print(f"--- {market} ---")
    print(df["label"].value_counts(normalize=True).round(3))
    print(df["barrier_hit"].value_counts(normalize=True).round(3))
    print(f"avg holding bars (all): {df['holding_bars'].mean():.1f}")
    print(f"avg holding bars (long/short only): {df.loc[df['label'] != 'no_trade', 'holding_bars'].mean():.1f}")
    print()

---

**Before uploading to Colab, look at:**
- **Class balance** — if `no_trade` dominates heavily (common with wide barriers / short holding periods), you'll want class weighting in the XGBoost objective, or to revisit `TP_MULT`/`SL_MULT`/`MAX_HOLDING_BARS` now rather than after training three models around it.
- **`barrier_hit` breakdown** — a high share of `session_end`/`timeout` vs `upper`/`lower` tells you how often the horizon is too short to let a real move play out.
- **`both_same_bar` share** — if this is more than a few percent, the same-bar approximation is affecting a non-trivial number of labels; worth knowing before you trust label quality near volatile bars.

Once this looks reasonable, `features_5min_labeled.csv` per market is what goes into each of the three Colab notebooks. Remember to still ordinal-encode `ctx_vol_regime` there (`low/normal/high` -> `0/1/2`) — it's still a plain string column in this output.